# 2-Step Retention Time Prediction — Overview & Prediction Pipeline

This project predicts **chromatographic retention times** in two steps:
1. **ROI prediction** — A message-passing neural network (DMPNN) predicts a *Retention Order Index* (ROI) for each compound using molecular structure + chromatographic system features.
2. **RT mapping** — The ROIs of anchor compounds (known retention times) are used to fit a regression model (LAD) that maps ROI → actual retention time.

Key files:
| File | Purpose |
|---|---|
| `predict.py` | End-to-end prediction from SMILES+metadata → RT |
| `train.py` | Train the ROI-prediction model |
| `mpnranker2.py` | Model definition + training loop |
| `utils_newbg.py` | `RankDataset` — pairwise/triplet dataset |
| `sampling.py` | Weighted sampling utilities |
| `evaluate.py` | Evaluation & model loading |
| `mapping.py` | LAD regression model (ROI → RT) |
| `dmpnn.py` / `dmpnn_graph.py` | DMPNN encoder implementation |

## Dependencies

In [ ]:
# Create conda environment (run once)
# !mamba env create -n 2-step -f ../env.yaml
# !mamba activate 2-step

# Core imports used throughout the project
import torch
import numpy as np
import pandas as pd
import yaml
import pickle
import json
import sys, os

# Make sure project root is on path
sys.path.insert(0, os.path.abspath('..'))

print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('MPS available:', torch.backends.mps.is_available())

## Loading a Pre-trained Model

In [ ]:
import sys
sys.path.insert(0, '..')

from evaluate import load_model

MODEL_PATH = '../models/2-step0525.pt'  # adjust if needed

model = load_model(MODEL_PATH, all_in_one=True)
model.eval()
print(type(model))
print('Max epoch trained:', model.max_epoch)

In [ ]:
# Patch missing dropout layers (needed for older saved models)
import torch.nn as nn

def patch_dropout(model):
    try:
        for enc in model.encoder.encoder:
            if not hasattr(enc, 'dropout_layer'):
                enc.dropout_layer = nn.Dropout(p=0.0)
        print('Dropout patch applied')
    except Exception as e:
        print('Patch failed:', e)

patch_dropout(model)

# Inspect what extra info is stored with the model
print('extra_storage keys:', list(model.extra_storage.keys()))

## Preparing Input Data

In [ ]:
# Example: load compounds TSV and metadata YAML
INPUT_COMPOUNDS = '../test/test_input.tsv'   # columns: smiles, rt (rt optional for unknowns)
INPUT_METADATA  = '../test/test_metadata.yaml'
REPO_ROOT       = '../../RepoRT/'            # path to RepoRT repo

# Show the metadata
metadata_raw = yaml.load(open(INPUT_METADATA), yaml.SafeLoader)
print(json.dumps(metadata_raw, indent=2))

In [ ]:
# Show the compounds file
compounds_df = pd.read_csv(INPUT_COMPOUNDS, sep='\t')
print(f'{len(compounds_df)} compounds')
compounds_df.head()

## Preprocessing (Feature Computation + Graph Construction)

In [ ]:
from utils import Data

data_args = model.extra_storage['data_args']
data_args['repo_root_folder'] = REPO_ROOT
sysfeature_scaler = model.extra_storage['sysfeature_scaler']

d = Data(**data_args)

# Flatten nested YAML metadata into dot-notation keys
[metadata] = pd.json_normalize(metadata_raw, sep='.').to_dict(orient='records')
print('Metadata keys:', list(metadata.keys()))

# Add external dataset (compounds with optional RTs as anchors)
original_columns = open(INPUT_COMPOUNDS).readlines()[0].strip().split('\t')
d.add_external_data(INPUT_COMPOUNDS, metadata=metadata,
                    remove_nan_rts=False, tab_mode=True,
                    isomeric=True, split_type='evaluate')
print(f'Loaded {len(d.df)} compounds')

In [ ]:
# Compute molecular descriptors and molecular graphs
d.compute_features(mode=None, add_descs=False)
d.compute_graphs()
d.split_data((0, 0))  # no train/val split needed for inference

if sysfeature_scaler is not None:
    d.standardize(other_descriptor_scaler=None,
                  other_sysfeature_scaler=sysfeature_scaler,
                  can_create_new_scaler=False)

((train_graphs, train_x, train_sys, train_y),
 (val_graphs,   val_x,   val_sys,   val_y),
 (test_graphs,  test_x,  test_sys,  test_y)) = d.get_split_data()

# Combine all splits (all go to 'evaluate')
X      = np.concatenate((train_x,   test_x,   val_x)).astype(np.float32)
X_sys  = np.concatenate((train_sys, test_sys, val_sys)).astype(np.float32)
Y      = np.concatenate((train_y,   test_y,   val_y))
graphs = np.concatenate((train_graphs, test_graphs, val_graphs))

print(f'X shape: {X.shape}, X_sys shape: {X_sys.shape}')

## Step 1 — ROI Prediction

In [ ]:
from time import time

t0 = time()
preds = model.predict(graphs, X, X_sys, batch_size=256, prog_bar=True)
print(f'Predicted {len(preds)} ROIs in {time()-t0:.2f}s')

# Restore original compound order
sort_order = np.argsort(np.concatenate([d.train_indices, d.test_indices, d.val_indices]))
d.df['roi'] = preds[sort_order]
d.df['roi2'] = d.df.roi ** 2  # for LAD model

d.df[['smiles', 'rt', 'roi']].head(10)

## Step 2 — RT Mapping via LAD Regression

In [ ]:
from mapping import LADModel

# Anchors: compounds with known retention time > void volume threshold
t0 = metadata['column.t0']
data_anchors    = d.df.loc[d.df.rt > t0]
data_to_predict = d.df.loc[pd.isna(d.df.rt)].copy()

print(f'Anchors: {len(data_anchors)}, To predict: {len(data_to_predict)}')

mapping_model = LADModel(data_anchors, ols_after=True,
                         ols_discard_if_negative=True, ols_drop_mode='2*median')

data_to_predict['rt_pred'] = mapping_model.get_mapping(data_to_predict.roi)
data_to_predict[['smiles', 'rt_pred']].head(10)

In [ ]:
# Visualise ROI vs RT for anchor compounds
import matplotlib.pyplot as plt

roi_range = np.linspace(data_anchors.roi.min(), data_anchors.roi.max(), 100)
rt_mapped = mapping_model.get_mapping(roi_range)

plt.figure(figsize=(7, 5))
plt.scatter(data_anchors.roi, data_anchors.rt, label='Anchors', s=30)
plt.plot(roi_range, rt_mapped, color='red', label='LAD mapping')
plt.xlabel('ROI (predicted)')
plt.ylabel('RT (min)')
plt.title('ROI → RT mapping')
plt.legend()
plt.tight_layout()
plt.show()

## CLI Usage Reference

```bash
# Prediction
python predict.py \
  --model models/2-step0525.pt \
  --repo_root_folder <path/to/RepoRT> \
  --input_compounds test/test_input.tsv \
  --input_metadata test/test_metadata.yaml \
  --out test/test_output.tsv

# With Docker
docker run -v $(pwd)/test:/app/test -v <RepoRT>:/RepoRT -it --rm \
  ghcr.io/boecker-lab/2-step:latest \
  python predict.py --model models/2-step0525.pt --repo_root_folder /RepoRT \
  --input_compounds test/test_input.tsv --input_metadata test/test_metadata.yaml
```